# LLM RAG demo + multicloud upload

Lightweight notebook showing:
- load prompts and target docs from experiments/data
- construct simple retrieval-augmented prompt (RAG) using local files
- call a local/mock LLM function (replace with real client)
- save predictions and upload to cloud via pca.multicloud.MultiCloudStore

In [ ]:
import json
from pathlib import Path
import yaml
from typing import Dict, Any, List

# change these imports if your package is installed differently
from pca import TfidfIndexer
from pca.poisoner import iter_corpus
from pca.multicloud import MultiCloudStore


## Configuration

In [ ]:
DATA_DIR = Path("experiments/data")
PROMPTS_FILE = DATA_DIR / "prompts" / "prompts_for_eval.yaml"
OUT_DIR = Path("experiments/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = OUT_DIR / "llm_preds.jsonl"


## Utility: simple mock LLM call — replace with real API (OpenAI, local Llama, etc.)

In [ ]:
def mock_llm_generate(prompt: str) -> str:
    # very naive placeholder: echo first sentence or summarize heuristically
    return prompt.split("\n")[0][:200] + " ... [mocked response]"


## Load prompts

In [ ]:
with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompts = yaml.safe_load(f)

prompts[:2]


## Build a TF-IDF index over the data for retrieval (uses pca.TfidfIndexer)

In [ ]:
indexer = TfidfIndexer()
indexer.build_from_directory(str(DATA_DIR))


## For each prompt, load the target file content and build a RAG-style input, then call LLM and write predictions.

In [ ]:
preds: List[Dict[str, Any]] = []
for p in prompts:
    pid = p.get("id")
    text = p.get("text", "")
    target_file = p.get("target_file")
    ctx = ""
    if target_file:
        tf = DATA_DIR / target_file
        if tf.exists():
            ctx = tf.read_text(encoding="utf-8")
    # simple RAG input: prompt + separator + context
    rag_input = f"{text}\n\nContext:\n{ctx}\n\nAnswer:"
    prediction = mock_llm_generate(rag_input)
    preds.append({"id": pid, "prompt": text, "prediction": prediction, "target_file": target_file})

with open(PRED_PATH, "w", encoding="utf-8") as out:
    for r in preds:
        out.write(json.dumps(r) + "\n")

print("Wrote predictions to", PRED_PATH)


## Optional: upload predictions to cloud. Fill provider, bucket, and remote_key and run the cell.

In [ ]:
# provider = "s3"  # or "gcs", "azure"
# bucket = "my-bucket"
# remote_key = "runs/demo/llm_preds.jsonl"
# store = MultiCloudStore(provider=provider, bucket=bucket)
# url = store.upload(str(PRED_PATH), remote_key)
# print("Uploaded to", url)


## Quick local inspection of saved preds

In [ ]:
with open(PRED_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(json.loads(line))
